In [0]:
from pyspark.sql.functions import current_date, current_timestamp, lit

medical_practitioners_source_path = (
    "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/health workforce/"
    "TABLE 9.14 Number of Government Medical Practitioners by Region- 2012 to 2023.csv"
)

# Preserve title, header, and data rows with Spark's positional column names.
source_file = medical_practitioners_source_path.rsplit('/', 1)[-1]
medical_practitioners_df = (
    spark.read
         .option("header", False)
         .option("inferSchema", False)
         .csv(medical_practitioners_source_path)
         .withColumn("source_file", lit(source_file))
         .withColumn("as_of_date", lit(None).cast("date"))
         .withColumn("extract_date", current_date())
         .withColumn("ingested_at", current_timestamp())
)

(
    medical_practitioners_df.write
                           .format("delta")
                           .mode("overwrite")
                           .option("delta.columnMapping.mode", "name")
                           .option("overwriteSchema", "true")
                           .saveAsTable(
                               "public_health.bronze.government_medical_practitioners_by_region"
                           )
)

display(medical_practitioners_df)